In [5]:
from pathlib import Path

RANDOM_STATE = 42

ZIP_PATH = Path('/content/Dataset 1-20260626T094526Z-3-001.zip')
EXTRACT_DIR = Path('/content/cicids2017_extracted')
OUTPUT_DIR = Path('/content/cicids2017_prepared')
OUTPUT_CSV_NAME = 'cicids2017_1771_leakage_safe.csv'

SOURCE_SPECS = [
    {
        'capture_group': 'ddos',
        'filename': 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
    },
    {
        'capture_group': 'portscan',
        'filename': 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
    },
    {
        'capture_group': 'webattacks',
        'filename': 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    },
]

SELECTION_PLAN = [
    ('ddos', 'BENIGN', 250),
    ('portscan', 'BENIGN', 250),
    ('webattacks', 'BENIGN', 250),
    ('ddos', 'DDoS', 250),
    ('portscan', 'PortScan', 250),
    ('webattacks', 'Web Attack - Brute Force', 250),
    ('webattacks', 'Web Attack - XSS', 250),
    ('webattacks', 'Web Attack - Sql Injection', 21),
]

EXCLUDE_PORTS_FROM_MODEL = True
EXCLUDE_PROTOCOL_FROM_MODEL = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [6]:
# =========================
# Imports and helper functions
# =========================
import re
import zipfile

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold


def read_cicids_csv(path: Path) -> pd.DataFrame:
    """Read CICIDS CSV robustly and normalise column headers."""
    try:
        frame = pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        frame = pd.read_csv(path, low_memory=False, encoding='latin1')

    frame.columns = (
        frame.columns.astype(str)
        .str.replace('\ufeff', '', regex=False)
        .str.strip()
    )
    return frame


def find_label_column(frame: pd.DataFrame) -> str:
    matches = [c for c in frame.columns if c.lower() == 'label']
    if len(matches) != 1:
        raise KeyError(f'Expected exactly one Label column; found: {matches}')
    return matches[0]


def normalize_label(value) -> str:
    """Normalise CICIDS label spelling without merging different attacks."""
    label = str(value).strip()
    label = label.replace(' ', '-').replace('–', '-').replace('—', '-')
    label = re.sub(r'\s*-\s*', ' - ', label)
    label = re.sub(r'\s+', ' ', label).strip()

    aliases = {
        'benign': 'BENIGN',
        'ddos': 'DDoS',
        'portscan': 'PortScan',
        'web attack - brute force': 'Web Attack - Brute Force',
        'web attack - xss': 'Web Attack - XSS',
        'web attack - sql injection': 'Web Attack - Sql Injection',
    }
    return aliases.get(label.lower(), label)


def resolve_source_file(root: Path, filename: str) -> Path:
    candidates = sorted(root.rglob(filename))
    if not candidates:
        raise FileNotFoundError(
            f'Could not find {filename!r} under {root}. '
            'Check ZIP_PATH or the content of SOURCE_SPECS.'
        )
    if len(candidates) > 1:
        print(f'Warning: multiple matches for {filename}; using {candidates[0]}')
    return candidates[0]


def numeric_feature_columns(frame: pd.DataFrame, excluded: set[str]) -> list[str]:
    """Convert genuine numeric flow fields and return their column names."""
    numeric_columns = []
    for column in frame.columns:
        if column in excluded:
            continue
        converted = pd.to_numeric(frame[column], errors='coerce')
        if converted.notna().mean() >= 0.95:
            frame[column] = converted
            numeric_columns.append(column)
    return numeric_columns


def available_group_count(frame: pd.DataFrame, group_col: str) -> int:
    return frame[group_col].nunique()


def safe_n_splits(y: pd.Series, groups: pd.Series, desired: int) -> int:
    """Return a stratified-group fold count supported by every target class."""
    counts = (
        pd.DataFrame({'target': y.to_numpy(), 'group': groups.to_numpy()})
        .drop_duplicates()
        .groupby('target')['group']
        .nunique()
    )
    if counts.empty or counts.min() < 2:
        raise ValueError('Each class needs at least two independent feature groups.')
    return int(min(desired, counts.min()))


def group_holdout(
    frame: pd.DataFrame,
    target_col: str,
    group_col: str,
    desired_splits: int,
    random_state: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Create a deterministic, label-stratified, group-disjoint holdout."""
    n_splits = safe_n_splits(frame[target_col], frame[group_col], desired_splits)
    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )
    left_idx, right_idx = next(
        splitter.split(frame, frame[target_col], groups=frame[group_col])
    )
    return frame.iloc[left_idx].copy(), frame.iloc[right_idx].copy()


In [7]:
if not EXTRACT_DIR.exists():
    if not ZIP_PATH.exists():
        raise FileNotFoundError(
            f'Archive not found: {ZIP_PATH}. Upload the ZIP or update ZIP_PATH.'
        )
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
        archive.extractall(EXTRACT_DIR)

frames = []
for spec in SOURCE_SPECS:
    source_file = resolve_source_file(EXTRACT_DIR, spec['filename'])
    frame = read_cicids_csv(source_file)
    label_column = find_label_column(frame)

    valid_label = frame[label_column].notna() & frame[label_column].astype(str).str.strip().ne('')
    dropped_unlabeled = int((~valid_label).sum())
    if dropped_unlabeled:
        print(f"Dropping {dropped_unlabeled:,} unlabeled rows from {source_file.name}")

    frame = frame.loc[valid_label].copy()
    frame['capture_group'] = spec['capture_group']
    frame['source_row_id'] = [f"{spec['capture_group']}::{i}" for i in frame.index]
    frame['label'] = frame[label_column].map(normalize_label)
    frame['target_binary'] = (frame['label'] != 'BENIGN').astype('int8')
    frames.append(frame)

raw_df = pd.concat(frames, ignore_index=True, sort=False)
raw_df = raw_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Loaded raw rows: {len(raw_df):,}')
print('\nRaw labels:')
print(raw_df['label'].value_counts().to_string())


Loaded raw rows: 682,578

Raw labels:
label
BENIGN                        393441
PortScan                      158930
DDoS                          128027
Web Attack - Brute Force        1507
Web Attack - XSS                 652
Web Attack - Sql Injection        21


In [8]:
IDENTIFIER_COLUMNS = {
    'Flow ID',
    'Source IP',
    'Destination IP',
    'Timestamp',
    'capture_group',
    'source_row_id',
    'label',
    'target_binary',
}

all_numeric_flow_features = numeric_feature_columns(raw_df, IDENTIFIER_COLUMNS)
if not all_numeric_flow_features:
    raise RuntimeError('No numeric CICIDS flow features found. Check the source CSV files.')

raw_df[all_numeric_flow_features] = raw_df[all_numeric_flow_features].replace(
    [np.inf, -np.inf], np.nan
)

rows_before_featureless_drop = len(raw_df)
raw_df = raw_df.dropna(subset=all_numeric_flow_features, how='all').copy()
featureless_rows_removed = rows_before_featureless_drop - len(raw_df)

raw_df['feature_fingerprint'] = pd.util.hash_pandas_object(
    raw_df[all_numeric_flow_features], index=False
).astype('uint64').astype(str)

rows_before_duplicate_removal = len(raw_df)
raw_df = raw_df.drop_duplicates(
    subset=all_numeric_flow_features + ['label'],
    keep='first',
).copy()
exact_duplicates_removed = rows_before_duplicate_removal - len(raw_df)

labels_per_fingerprint = raw_df.groupby('feature_fingerprint')['label'].nunique()
conflicting_fingerprints = labels_per_fingerprint[labels_per_fingerprint > 1].index
conflicting_rows_removed = int(raw_df['feature_fingerprint'].isin(conflicting_fingerprints).sum())
raw_df = raw_df.loc[~raw_df['feature_fingerprint'].isin(conflicting_fingerprints)].copy()

print('Data-quality summary')
print(f'  featureless rows removed: {featureless_rows_removed:,}')
print(f'  exact feature+label duplicates removed: {exact_duplicates_removed:,}')
print(f'  contradictory-label rows removed: {conflicting_rows_removed:,}')
print(f'  rows remaining after cleaning: {len(raw_df):,}')
print(f'  numeric flow features found: {len(all_numeric_flow_features):,}')


Data-quality summary
  featureless rows removed: 0
  exact feature+label duplicates removed: 86,332
  contradictory-label rows removed: 2
  rows remaining after cleaning: 596,244
  numeric flow features found: 78


In [9]:
selected_parts = []
selection_audit = []

for plan_index, (capture_group, label, required_rows) in enumerate(SELECTION_PLAN, start=1):
    candidates = raw_df.loc[
        (raw_df['capture_group'] == capture_group) &
        (raw_df['label'] == label)
    ].copy()

    independent_groups = candidates['feature_fingerprint'].nunique()
    if independent_groups < required_rows:
        raise ValueError(
            f'Not enough independent rows for ({capture_group}, {label}). '
            f'Required {required_rows}, available {independent_groups}. '
            'Do not oversample or fabricate rows; inspect the raw source data instead.'
        )

    chosen = candidates.sample(
        n=required_rows,
        replace=False,
        random_state=RANDOM_STATE + plan_index,
    )
    selected_parts.append(chosen)
    selection_audit.append({
        'capture_group': capture_group,
        'label': label,
        'required_rows': required_rows,
        'available_independent_rows': independent_groups,
        'selected_rows': len(chosen),
    })

selected_df = pd.concat(selected_parts, ignore_index=True)
selected_df = selected_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
selected_df['row_id'] = [f'cicids1771::{i:04d}' for i in range(len(selected_df))]

EXPECTED_TOTAL = sum(required_rows for _, _, required_rows in SELECTION_PLAN)
assert EXPECTED_TOTAL == 1771, f'Plan must total 1771, got {EXPECTED_TOTAL}'
assert len(selected_df) == EXPECTED_TOTAL, 'Selected-row count does not match the requested total.'
assert selected_df['feature_fingerprint'].is_unique, 'Duplicate feature groups survived selection.'

print(pd.DataFrame(selection_audit).to_string(index=False))
print('\nSelected class counts:')
print(selected_df['label'].value_counts().to_string())
print(f'\nExact selected total: {len(selected_df):,}')


capture_group                      label  required_rows  available_independent_rows  selected_rows
         ddos                     BENIGN            250                       93653            250
     portscan                     BENIGN            250                      121348            250
   webattacks                     BENIGN            250                      160266            250
         ddos                       DDoS            250                      128015            250
     portscan                   PortScan            250                       90819            250
   webattacks   Web Attack - Brute Force            250                        1470            250
   webattacks           Web Attack - XSS            250                         652            250
   webattacks Web Attack - Sql Injection             21                          21             21

Selected class counts:
label
BENIGN                        750
DDoS                          250
Web Attack 

In [10]:
train_val_df, test_df = group_holdout(
    selected_df,
    target_col='label',
    group_col='feature_fingerprint',
    desired_splits=7,
    random_state=RANDOM_STATE,
)
train_df, validation_df = group_holdout(
    train_val_df,
    target_col='label',
    group_col='feature_fingerprint',
    desired_splits=6,
    random_state=RANDOM_STATE + 1,
)

split_frames = {
    'train': train_df,
    'validation': validation_df,
    'test': test_df,
}

split_names = list(split_frames)
for i, left_name in enumerate(split_names):
    for right_name in split_names[i + 1:]:
        left_groups = set(split_frames[left_name]['feature_fingerprint'])
        right_groups = set(split_frames[right_name]['feature_fingerprint'])
        overlap = left_groups & right_groups
        assert not overlap, (
            f'Feature-group leakage between {left_name} and {right_name}: {len(overlap)} groups'
        )

final_frames = []
for split_name, split_df in split_frames.items():
    piece = split_df.copy()
    piece.insert(0, 'split', split_name)
    final_frames.append(piece)

final_df = pd.concat(final_frames, ignore_index=True)
final_df = final_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
assert len(final_df) == 1771

summary = pd.DataFrame([
    {
        'split': name,
        'rows': len(frame),
        'unique_feature_groups': frame['feature_fingerprint'].nunique(),
        'labels': frame['label'].nunique(),
        'attack_ratio': round(float(frame['target_binary'].mean()), 4),
    }
    for name, frame in split_frames.items()
])
print(summary.to_string(index=False))


     split  rows  unique_feature_groups  labels  attack_ratio
     train  1265                   1265       6        0.5787
validation   253                    253       6        0.6008
      test   253                    253       6        0.5415


In [12]:
MODEL_EXCLUSIONS = {
    'Flow ID',
    'Source IP',
    'Destination IP',
    'Timestamp',
    'capture_group',
    'source_row_id',
    'feature_fingerprint',
    'row_id',
    'label',
    'target_binary',
}
if EXCLUDE_PORTS_FROM_MODEL:
    MODEL_EXCLUSIONS.update({'Source Port', 'Destination Port'})
if EXCLUDE_PROTOCOL_FROM_MODEL:
    MODEL_EXCLUSIONS.add('Protocol')

MODEL_FEATURES = [
    column for column in all_numeric_flow_features
    if column not in MODEL_EXCLUSIONS
]
if not MODEL_FEATURES:
    raise RuntimeError('No model features remain after leakage exclusions.')

export_columns = ['split', 'row_id', 'label', 'target_binary'] + MODEL_FEATURES
export_df = final_df[export_columns].copy()

assert len(export_df) == 1771
assert export_df['row_id'].is_unique
assert export_df['label'].isna().sum() == 0
assert export_df[MODEL_FEATURES].shape[1] > 0

output_path = OUTPUT_DIR / OUTPUT_CSV_NAME
export_df.to_csv(output_path, index=False)

print(f'Created exactly one CSV: {output_path}')
print(f'Rows: {len(export_df):,}')
print(f'Model features: {len(MODEL_FEATURES):,}')
print('\nClass distribution:')
print(export_df['label'].value_counts().to_string())
print('\nHow to train safely:')
print("  1) train rows:      split == 'train'")
print("  2) validation rows: split == 'validation'")
print("  3) test rows:       split == 'test'")
print("  4) Inputs: MODEL_FEATURES only; do not use split, row_id, label, or target_binary as inputs.")

Created exactly one CSV: /content/cicids2017_prepared/cicids2017_1771_leakage_safe.csv
Rows: 1,771
Model features: 77

Class distribution:
label
BENIGN                        750
PortScan                      250
Web Attack - Brute Force      250
DDoS                          250
Web Attack - XSS              250
Web Attack - Sql Injection     21

How to train safely:
  1) train rows:      split == 'train'
  2) validation rows: split == 'validation'
  3) test rows:       split == 'test'
  4) Inputs: MODEL_FEATURES only; do not use split, row_id, label, or target_binary as inputs.
